In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------
DATA_PATH = Path("data/200thrillers.csv")
# Switch to Path("data/thrillers.csv") if you want the thriller-only corpus.
TITLE_COL = "original_title"
DESCRIPTION_COL = "description"
AUTHOR_COL = "author"
TOP_N = 5
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CACHE_DIR = Path(".cache/book_similarity")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Change this to test different prompts.
QUERY_TEXT = (
    "A dark psychological thriller about a missing person, hidden secrets, "
    "and a tense investigation that keeps revealing disturbing details."
)


# -----------------------------------------------------------------------------
# LOAD + CLEAN CORPUS
# -----------------------------------------------------------------------------
def load_corpus(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    if DESCRIPTION_COL not in df.columns:
        raise KeyError(f"Missing required column: {DESCRIPTION_COL}")
    if TITLE_COL not in df.columns:
        raise KeyError(f"Missing required column: {TITLE_COL}")

    df = df.copy()
    df[DESCRIPTION_COL] = df[DESCRIPTION_COL].fillna("").astype(str).str.strip()
    df[TITLE_COL] = df[TITLE_COL].fillna("Unknown Title").astype(str).str.strip()

    if AUTHOR_COL in df.columns:
        df[AUTHOR_COL] = df[AUTHOR_COL].fillna("").astype(str).str.strip()

    df = df[df[DESCRIPTION_COL] != ""].reset_index(drop=True)
    print(f"Loaded {len(df):,} books from {path}")
            "source": [
                "# -----------------------------------------------------------------------------",
                "# TEST MULTIPLE QUERIES WITH CACHED EMBEDDINGS",
                "# -----------------------------------------------------------------------------",
                "df = load_corpus(DATA_PATH)",
                "",
                "MODEL_CANDIDATES = [",
                "    \"all-MiniLM-L6-v2\",",
                "    \"BAAI/bge-small-en-v1.5\",",
                "    \"sentence-transformers/all-mpnet-base-v2\",",
                "]",
                "",
                "QUERY_TEXTS = [",
                "    \"A psychological thriller about marital issues\",",
                "    \"A dark mystery about a missing person and hidden secrets\",",
                "    \"A historical romance set in 20th century England\",",
                "]",
                "",
                "for query_text in QUERY_TEXTS:",
                "    print(f\"\\n{'=' * 80}\")",
                "    print(f\"Query: {query_text}\")",
                "",
                "    for candidate_model in MODEL_CANDIDATES:",
                "        candidate_embeddings = build_embeddings(df, candidate_model, DATA_PATH)",
                "        candidate_results = search_books(",
                "            query_text,",
                "            df,",
                "            candidate_embeddings,",
                "            candidate_model,",
                "            top_n=TOP_N,",
                "        )",
                "",
                "        print(f\"\\nModel: {candidate_model}\")",
                "        print(candidate_results.to_string(index=False))",
                ""
            ]
    print(f"Encoding {len(texts):,} descriptions...")
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    np.save(embeddings_path, embeddings)
    meta_path.write_text(
        json.dumps(
            {
                "model_name": model_name,
                "source_path": str(path),
                "rows": len(df),
                "text_column": DESCRIPTION_COL,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Saved embeddings to {embeddings_path}")
    return embeddings


# -----------------------------------------------------------------------------
# COSINE SIMILARITY SEARCH
# -----------------------------------------------------------------------------
def search_books(
    query_text: str,
    df: pd.DataFrame,
    embeddings: np.ndarray,
    model_name: str,
    top_n: int = 5,
) -> pd.DataFrame:
    if not query_text.strip():
        raise ValueError("query_text cannot be empty")

    model = SentenceTransformer(model_name)
    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_n]

    results = df.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]
    results["rank"] = range(1, len(results) + 1)

    columns = ["rank", TITLE_COL, "similarity"]
    if AUTHOR_COL in results.columns:
        columns.insert(2, AUTHOR_COL)
    if DESCRIPTION_COL in results.columns:
        columns.append(DESCRIPTION_COL)

    return results[columns]


In [ ]:
# -----------------------------------------------------------------------------
# RUN DEMO
# -----------------------------------------------------------------------------
df = load_corpus(DATA_PATH)
embeddings = build_embeddings(df, MODEL_NAME, DATA_PATH)
results = search_books(QUERY_TEXT, df, embeddings, MODEL_NAME, top_n=TOP_N)

print("\nQuery:")
print(QUERY_TEXT)
print("\nTop matches:")
print(results.to_string(index=False))


In [ ]:
QUERY_TEXT = (
    "A butler's love story set in 20th century England."
)
results = search_books(QUERY_TEXT, df, embeddings, MODEL_NAME, top_n=TOP_N)

print("\nQuery:")
print(QUERY_TEXT)
print("\nTop matches:")
print(results.to_string(index=False))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2711.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query:
A dark psychological thriller about a missing person, hidden secrets, and a tense investigation that keeps revealing disturbing details.

Top matches:
 rank  original_title          author  similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [4]:
import time

df = load_corpus(DATA_PATH)

QUERY_TEXT = (
    "A psychological thriller about marital issues"
)

MODEL_CANDIDATES = [
    "all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "sentence-transformers/all-mpnet-base-v2",
]

benchmark_rows = []
all_results = {}

for candidate_model in MODEL_CANDIDATES:
    print(f"\n{'=' * 80}")
    print(f"Testing model: {candidate_model}")

    t0 = time.perf_counter()
    candidate_embeddings = build_embeddings(df, candidate_model, DATA_PATH)
    embed_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    candidate_results = search_books(
        QUERY_TEXT,
        df,
        candidate_embeddings,
        candidate_model,
        top_n=TOP_N,
    )
    search_time = time.perf_counter() - t1

    all_results[candidate_model] = candidate_results

    benchmark_rows.append(
        {
            "model": candidate_model,
            "embedding_dim": int(candidate_embeddings.shape[1]),
            "corpus_size": int(candidate_embeddings.shape[0]),
            "embed_time_sec": round(embed_time, 3),
            "query_time_sec": round(search_time, 4),
            "top1_similarity": float(candidate_results.iloc[0]["similarity"]),
        }
    )

    print("Top matches:")
    print(candidate_results.to_string(index=False))

benchmark_df = pd.DataFrame(benchmark_rows).sort_values("top1_similarity", ascending=False)

print(f"\n{'=' * 80}")
print("Model benchmark summary")
print(benchmark_df.to_string(index=False))

Loaded 200 books from 200thrillers.csv

Testing model: all-MiniLM-L6-v2
Loading transformer model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1300.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 200 descriptions...


Batches: 100%|██████████| 7/7 [00:10<00:00,  1.52s/it]


Saved embeddings to .cache\book_similarity\200thrillers__all-MiniLM-L6-v2.npy


...

Encoding 200 descriptions...


Batches: 100%|██████████| 7/7 [01:26<00:00, 12.29s/it]


Saved embeddings to .cache\book_similarity\200thrillers__sentence-transformers__all-mpnet-base-v2.npy


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1530.46it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top matches:
 rank      original_title  similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [5]:
# -----------------------------------------------------------------------------
# TEST MULTIPLE QUERIES WITH CACHED EMBEDDINGS
# -----------------------------------------------------------------------------
QUERY_TEXTS = [
    "A psychological thriller about marital issues"
]

for query_text in QUERY_TEXTS:
    print(f"\n{'=' * 80}")
    print(f"Query: {query_text}")

    for candidate_model in MODEL_CANDIDATES:
        candidate_embeddings = build_embeddings(df, candidate_model, DATA_PATH)
        candidate_results = search_books(
            query_text,
            df,
            candidate_embeddings,
            candidate_model,
            top_n=TOP_N,
        )

        print(f"\nModel: {candidate_model}")
        print(candidate_results.to_string(index=False))



Query: A psychological thriller about marital issues
Using cached embeddings from .cache\book_similarity\200thrillers__all-MiniLM-L6-v2.npy


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 653.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model: all-MiniLM-L6-v2
 rank       original_title  similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 730.13it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model: BAAI/bge-small-en-v1.5
 rank    original_title  similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2044.60it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model: sentence-transformers/all-mpnet-base-v2
 rank      original_title  similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   